# Ollama + GroundingDINO GPU Server on Google Colab (Receipt AI)

Questo notebook permette di eseguire **Ollama** e **GroundingDINO** su **GPU T4 gratuita** ed esporli al backend locale tramite **Cloudflare Tunnel**.

### Istruzioni rapide
1. Vai su **Runtime -> Change runtime type** e seleziona **T4 GPU**.
2. Scegli il modello nella cella 3 (es. `gemma4:e2b`).
3. Esegui le celle in ordine (`Run all`).
4. Copia i due URL `https://....trycloudflare.com` stampati dall'ultima cella e incollali nel tuo file `backend/.env`:
   ```env
   OLLAMA_URL=[https://xxxx-xxxx.trycloudflare.com](https://xxxx-xxxx.trycloudflare.com)
   OLLAMA_MODEL=gemma4:e2b
   RECEIPT_DETECTOR_PROVIDER=remote
   DETECTOR_API_URL=[https://yyyy-yyyy.trycloudflare.com](https://yyyy-yyyy.trycloudflare.com)
   ```
5. Riavvia il backend locale (`uvicorn`).

In [ ]:
# 1. Verifica che la GPU T4 sia attiva
!nvidia-smi

In [ ]:
# 2. Installa zstd, pciutils (lspci per far rilevare la GPU a Ollama), Ollama e cloudflared
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!rm -f cloudflared-linux-amd64.deb

In [ ]:
#@title 3. Avvia Ollama e scarica il modello
MODEL_NAME = "gemma4:e2b" #@param {type:"string"}
MODEL_NAME = MODEL_NAME.strip() or "gemma4:e2b"

import os, subprocess, time, urllib.request

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_ORIGINS"] = "*"

subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)

# Attendi che Ollama sia pronto
for _ in range(20):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama attivo!")
        break
    except Exception:
        time.sleep(1)

print(f"Download del modello {MODEL_NAME}...")
!ollama pull {MODEL_NAME}

In [ ]:
#@title 4. Avvia GroundingDINO (porta 8001) + Tunnel Cloudflare per Ollama e Detector
!pip -q install fastapi uvicorn python-multipart transformers torch pillow

import io, re, subprocess, threading, time
import torch
import uvicorn
from fastapi import FastAPI, File, UploadFile
from PIL import Image, ImageOps
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

MODEL_ID = "IDEA-Research/grounding-dino-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Caricamento {MODEL_ID} su {DEVICE}...")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()
print("GroundingDINO pronto!")

det_app = FastAPI()

@det_app.post("/detect")
async def detect(file: UploadFile = File(...)):
    raw = await file.read()
    img = ImageOps.exif_transpose(Image.open(io.BytesIO(raw))).convert("RGB")
    w, h = img.size

    inputs = processor(images=img, text="receipt . ticket . invoice .", return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)

    try:
        results = processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids, threshold=0.25, text_threshold=0.25, target_sizes=[(h, w)]
        )[0]
    except TypeError:
        results = processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids, box_threshold=0.25, text_threshold=0.25, target_sizes=[(h, w)]
        )[0]

    raw_boxes = []
    for box, score in zip(results["boxes"].tolist(), results["scores"].tolist()):
        x1, y1, x2, y2 = box
        left = max(0, min(int(round(x1)), w - 1))
        top = max(0, min(int(round(y1)), h - 1))
        right = max(left + 1, min(int(round(x2)), w))
        bottom = max(top + 1, min(int(round(y2)), h))
        bw, bh = right - left, bottom - top
        if bw >= max(20, int(w * 0.05)) and bh >= max(20, int(h * 0.05)):
            raw_boxes.append((left, top, bw, bh, float(score)))

    raw_boxes.sort(key=lambda b: b[4], reverse=True)
    deduped = []
    for x1, y1, w1, h1, _ in raw_boxes:
        keep = True
        for x2, y2, w2, h2 in deduped:
            ix1, iy1 = max(x1, x2), max(y1, y2)
            ix2, iy2 = min(x1 + w1, x2 + w2), min(y1 + h1, y2 + h2)
            inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
            union = w1 * h1 + w2 * h2 - inter
            if union > 0 and (inter / union) > 0.5:
                keep = False
                break
        if keep:
            deduped.append((x1, y1, w1, h1))

    def contains(o, i):
        ox, oy, ow, oh = o
        ix, iy, iw, ih = i
        return (
            ix >= ox - 12
            and iy >= oy - 12
            and ix + iw <= ox + ow + 12
            and iy + ih <= oy + oh + 12
            and (ow * oh) > (iw * ih) * 1.4
        )

    final_boxes = [b for b in deduped if sum(1 for o in deduped if o != b and contains(b, o)) < 2]
    row_tol = max(20, int(h * 0.08))
    final_boxes.sort(key=lambda b: (b[1] // row_tol, b[0]))

    return {
        "image_width": w,
        "image_height": h,
        "boxes": [{"x": x, "y": y, "width": bw, "height": bh, "page": 0} for x, y, bw, bh in final_boxes],
    }

threading.Thread(
    target=lambda: uvicorn.run(det_app, host="0.0.0.0", port=8001, log_level="warning"),
    daemon=True,
).start()
time.sleep(2)

def run_tunnel(port: int, env_key: str):
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            print(f"\n{'='*60}\nCOPIA NEL TUO .env:\n{env_key}={m.group(0)}\n{'='*60}\n", flush=True)

threading.Thread(target=run_tunnel, args=(11434, "OLLAMA_URL"), daemon=True).start()
threading.Thread(target=run_tunnel, args=(8001, "DETECTOR_API_URL"), daemon=True).start()

print("Tunnel avviati! Lascia questa cella in esecuzione.")
while True:
    time.sleep(60)